# Práctica — Clasificación y Métodos de Ensamble

## Objetivo

Esta práctica no busca solamente ejecutar código. Busca observar cómo se comportan distintos modelos de clasificación y justificar decisiones como ingenieros.

Vamos a trabajar tres ideas:

1. Los árboles de decisión pueden ser inestables.
2. Random Forest estabiliza combinando muchos árboles.
3. La elección del “mejor” modelo depende de la métrica y del costo del error.

---

## Setup

Ejecutá esta celda primero. Si `xgboost` no está instalado, el notebook usa una alternativa de `scikit-learn` para mantener la práctica funcionando.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

print("XGBoost disponible:", XGBOOST_AVAILABLE)


# Parte A — Experimentar la inestabilidad del árbol

## Concepto

Un árbol de decisión puede cambiar mucho si cambia levemente el dataset.  
Esto es una manifestación de **alta varianza** o **varianza estructural**.

Vamos a entrenar 10 árboles sobre muestras bootstrap del mismo dataset y ver qué pasa con:

- la raíz del árbol
- la predicción de un punto ambiguo

---

In [ ]:
# Dataset pequeño de riesgo crediticio
# Ing = ingreso, Deu = deuda, R = riesgo
data = pd.DataFrame({
    "Ingreso": [2, 3, 4, 5, 6, 7, 8, 9, 4, 6],
    "Deuda":   [8, 7, 6, 5, 4, 3, 2, 1, 7, 6],
    "Riesgo":  [1, 1, 1, 0, 0, 0, 0, 0, 1, 1]
})

X = data[["Ingreso", "Deuda"]]
y = data["Riesgo"]

punto_ambiguo = pd.DataFrame({"Ingreso": [6], "Deuda": [6]})

data


In [ ]:
def entrenar_arboles_bootstrap(X, y, n_modelos=10, random_state=42):
    resultados = []

    for i in range(n_modelos):
        X_boot, y_boot = resample(
            X, y,
            replace=True,
            n_samples=len(X),
            random_state=random_state + i
        )

        tree = DecisionTreeClassifier(max_depth=None, random_state=random_state + i)
        tree.fit(X_boot, y_boot)

        pred = tree.predict(punto_ambiguo)[0]
        root_feature = X.columns[tree.tree_.feature[0]] if tree.tree_.feature[0] != -2 else "Hoja"

        resultados.append({
            "modelo": i + 1,
            "raiz": root_feature,
            "prediccion_punto_ambiguo": pred
        })

    return pd.DataFrame(resultados)

resultados_arboles = entrenar_arboles_bootstrap(X, y)
resultados_arboles


In [ ]:
print("Distribución de predicciones:")
print(resultados_arboles["prediccion_punto_ambiguo"].value_counts())

print("\nRaíces utilizadas:")
print(resultados_arboles["raiz"].value_counts())

resultados_arboles["prediccion_punto_ambiguo"].value_counts().plot(kind="bar")
plt.title("Parte A — Predicciones de árboles individuales")
plt.xlabel("Predicción")
plt.ylabel("Cantidad de modelos")
plt.show()


## Reflexión Parte A

Respondé brevemente:

1. ¿Todos los árboles predijeron lo mismo para el punto ambiguo?
2. ¿La raíz del árbol fue siempre la misma?
3. ¿Qué nos dice esto sobre la estabilidad de los árboles individuales?

**Respuesta:**  
_Escribir aquí._

# Parte B — Observar la estabilización con Random Forest

## Concepto

Random Forest combina muchos árboles.  
La idea no es que cada árbol sea estable, sino que el conjunto sea más estable que cada árbol individual.

---

In [ ]:
def entrenar_random_forests_bootstrap(X, y, n_modelos=10, random_state=42):
    resultados = []

    for i in range(n_modelos):
        X_boot, y_boot = resample(
            X, y,
            replace=True,
            n_samples=len(X),
            random_state=random_state + i
        )

        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            random_state=random_state + i
        )
        rf.fit(X_boot, y_boot)

        pred = rf.predict(punto_ambiguo)[0]

        resultados.append({
            "modelo": i + 1,
            "prediccion_punto_ambiguo": pred
        })

    return pd.DataFrame(resultados)

resultados_rf = entrenar_random_forests_bootstrap(X, y)
resultados_rf


In [ ]:
print("Distribución de predicciones — Árboles individuales:")
print(resultados_arboles["prediccion_punto_ambiguo"].value_counts())

print("\nDistribución de predicciones — Random Forest:")
print(resultados_rf["prediccion_punto_ambiguo"].value_counts())

comparacion = pd.DataFrame({
    "Decision Tree": resultados_arboles["prediccion_punto_ambiguo"].value_counts(),
    "Random Forest": resultados_rf["prediccion_punto_ambiguo"].value_counts()
}).fillna(0)

comparacion.plot(kind="bar")
plt.title("Comparación de estabilidad: Árbol vs Random Forest")
plt.xlabel("Predicción")
plt.ylabel("Cantidad de modelos")
plt.show()


## Reflexión Parte B

Respondé brevemente:

1. ¿Random Forest fue más estable que los árboles individuales?
2. ¿Qué cambió entre la Parte A y la Parte B?
3. ¿Por qué promediar modelos puede reducir varianza?

**Respuesta:**  
_Escribir aquí._

# Parte C — Tomar decisiones como ingenieros

## Concepto

Ahora vamos a comparar modelos usando métricas de clasificación.

Modelos:

- Decision Tree
- Random Forest
- XGBoost, si está instalado
- Gradient Boosting como alternativa si XGBoost no está disponible

La pregunta no es solo: “¿qué modelo tiene mejor número?”  
La pregunta es: **¿qué modelo elegirías según el costo del error?**

---

## Dataset

Este notebook intenta usar un archivo `titanic.csv` si está en la misma carpeta.

Columnas esperadas, si usás Titanic real:

- `Survived`
- `Pclass`
- `Sex`
- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Embarked`

Si el archivo no existe, el notebook genera un dataset sintético con estructura similar para que la práctica pueda ejecutarse igual.

In [ ]:
from pathlib import Path

def cargar_o_generar_titanic():
    path = Path("titanic.csv")

    if path.exists():
        df = pd.read_csv(path)
        print("Usando titanic.csv local")
        return df

    print("No se encontró titanic.csv. Generando dataset sintético tipo Titanic.")

    n = 900
    rng = np.random.default_rng(RANDOM_STATE)

    pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
    sex = rng.choice(["male", "female"], size=n, p=[0.62, 0.38])
    age = np.clip(rng.normal(30, 14, size=n), 1, 80)
    fare = np.clip(rng.lognormal(mean=3.0, sigma=0.8, size=n), 5, 250)
    sibsp = rng.poisson(0.5, size=n)
    parch = rng.poisson(0.35, size=n)
    embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.18, 0.10])

    # Probabilidad sintética de supervivencia
    logit = (
        -1.2
        + 1.4 * (sex == "female")
        + 0.75 * (pclass == 1)
        + 0.25 * (pclass == 2)
        - 0.018 * age
        + 0.004 * fare
        - 0.15 * sibsp
        - 0.10 * parch
    )

    prob = 1 / (1 + np.exp(-logit))
    survived = rng.binomial(1, prob)

    return pd.DataFrame({
        "Survived": survived,
        "Pclass": pclass,
        "Sex": sex,
        "Age": age,
        "SibSp": sibsp,
        "Parch": parch,
        "Fare": fare,
        "Embarked": embarked
    })

df = cargar_o_generar_titanic()
df.head()


In [ ]:
# Preparación básica
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
target = "Survived"

df_model = df[features + [target]].copy()
df_model = df_model.dropna()

X = pd.get_dummies(df_model[features], drop_first=True)
y = df_model[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Distribución de clases en test:")
print(y_test.value_counts(normalize=True))


In [ ]:
# Definición de modelos
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )
else:
    models["Gradient Boosting (fallback)"] = GradientBoostingClassifier(random_state=RANDOM_STATE)

models


In [ ]:
def evaluar_modelo(nombre, modelo, X_train, X_test, y_train, y_test, threshold=0.5):
    modelo.fit(X_train, y_train)

    if hasattr(modelo, "predict_proba"):
        proba = modelo.predict_proba(X_test)[:, 1]
    else:
        proba = modelo.decision_function(X_test)

    y_pred = (proba >= threshold).astype(int)

    return {
        "Modelo": nombre,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_test, proba)
    }

resultados = []
for nombre, modelo in models.items():
    resultados.append(evaluar_modelo(nombre, modelo, X_train, X_test, y_train, y_test, threshold=0.5))

resultados_df = pd.DataFrame(resultados).sort_values("AUC", ascending=False)
resultados_df


## Experimento con thresholds

Ahora vamos a ver cómo cambia la decisión al modificar el umbral.

Probamos:

- 0.3
- 0.5
- 0.7

Observá especialmente qué pasa con **precision** y **recall**.

In [ ]:
thresholds = [0.3, 0.5, 0.7]

resultados_thresholds = []

for threshold in thresholds:
    for nombre, modelo in models.items():
        resultados_thresholds.append(
            evaluar_modelo(nombre, modelo, X_train, X_test, y_train, y_test, threshold=threshold)
        )

thresholds_df = pd.DataFrame(resultados_thresholds)
thresholds_df.sort_values(["Modelo", "Threshold"])


In [ ]:
# Visualización simple de Precision y Recall por threshold
for nombre in thresholds_df["Modelo"].unique():
    subset = thresholds_df[thresholds_df["Modelo"] == nombre]

    plt.figure()
    plt.plot(subset["Threshold"], subset["Precision"], marker="o", label="Precision")
    plt.plot(subset["Threshold"], subset["Recall"], marker="o", label="Recall")
    plt.title(f"Precision vs Recall según threshold — {nombre}")
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.legend()
    plt.show()


## Escenario 1 — Equipo de rescate

Supongamos que el modelo se usa para priorizar rescates.

**Objetivo:** encontrar a todas las personas vivas.

- Error más grave: Falso Negativo  
- Métrica prioritaria: Recall

### Pregunta

¿Qué modelo y qué threshold elegirías si querés maximizar recall?

Justificá con métricas.

**Respuesta:**  
_Escribir aquí._

## Escenario 2 — Uso eficiente de recursos

Supongamos que los recursos de rescate son muy limitados.

**Objetivo:** evitar enviar recursos a objetivos incorrectos.

- Error más grave: Falso Positivo  
- Métrica prioritaria: Precision

### Pregunta

¿Qué modelo y qué threshold elegirías si querés maximizar precision?

Justificá con métricas.

**Respuesta:**  
_Escribir aquí._

## Comparación final

Respondé:

1. ¿Elegiste el mismo modelo en ambos escenarios?
2. ¿Elegiste el mismo threshold?
3. ¿Qué métrica fue más útil en cada caso?
4. ¿Qué modelo tuvo mejor AUC?
5. ¿El modelo con mejor AUC fue necesariamente el mejor para cada escenario?

**Respuesta:**  
_Escribir aquí._

# Cierre

La práctica muestra tres ideas:

1. Un árbol individual puede ser inestable.
2. Random Forest estabiliza combinando árboles.
3. Elegir un modelo no depende de una sola métrica universal.

## Idea final

> No existe el mejor modelo en abstracto.  
> Existe el mejor modelo para un contexto.
